In [2]:
import sqlite3
import pandas as pd
import os
from datetime import datetime




In [3]:
conexao = sqlite3.connect(database = 'db_project_eng_dados')

In [10]:
# Camada Bronze:
df = pd.read_csv('../landing/z0019_1.csv')
# Como já está separado por vírgulas, não há necessidade colocar esse parâmetro.
# Se o arquivo estivesse separados por ponto e vírgula, aí sim seria necessário. ()'sep = ',')

In [13]:
df.head(2)

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10m,SIRGAS 2000 / UTM 23S,1500,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22 00:38:37.735803
1,2,Modelo Digital de Elevação,MDE,Raster,30m,WGS84,5000,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22 00:38:37.735803


In [4]:
# Dicionário de dados:
# Pode salvar como: dicionario_bronze_produtos.csv
# Ótimo para documentação do projeto ou README

dict_data = pd.read_csv('../landing/dict_bronze_prod.csv')







In [6]:
print(dict_data)

                 coluna  tipo                          porque
0            id_produto  TEXT  SQLite trata VARCHAR como TEXT
1          nome_produto  TEXT                     Texto livre
2             categoria  TEXT                   Classificação
3             tipo_dado  TEXT              Sem enum no Bronze
4    resolucao_espacial  REAL      Valor numérico com decimal
5   sistema_coordenadas  TEXT                    EPSG é texto
6    area_cobertura_km2  REAL             Permite soma, média
7        data_aquisicao  TEXT      ISO 8601 (padrão em dados)
8               formato  TEXT                          String
9            fornecedor  TEXT                          String
10                preco  REAL                Cálculos futuros
11          data_bronze  TEXT            Auditoria / linhagem


In [11]:
arquivo = 'z0019_1.csv'
data_ingestion = datetime.now()
df['data_bronze'] = data_ingestion
#df.head(2)

In [17]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [7]:
conexao.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos(
                id_produto TEXT, 
                nome_produto TEXT,
                categoria TEXT,
                tipo_dado TEXT,
                resolucao_espacial REAL,
                sistema_coordenadas TEXT,
                area_cobertura_km2 REAL,
                data_aquisicao TEXT,
                formato TEXT,
                fornecedor TEXT,
                preco REAL,
                data_bronze TEXT) """)

In [ ]:
# SQLite não “enxerga” DataFrames diretamente
# O df só existe na memória do Python, não no engine SQL do SQLite
# No DuckDB o DataFrame vira uma view temporária automaticamente.
# No SQLite, você precisa enviar os dados do DataFrame para a tabela via Python.
# Forma mais simples e robusta.

df.to_sql( 'bronze_produtos', conexao, if_exists='append', index=False)

#DataFrame (Python)
#   ↓
#to_sql (append)
#   ↓
#SQLite (bronze_produtos)


7

In [ ]:
# Isso é ótimo para APIs, mas ruim para visualização.
resultado = conexao.execute("SELECT * FROM bronze_produtos").fetchmany()

In [ ]:
# Formato padrão tabular:
df_resultado = pd.read_sql_query ("SELECT * FROM bronze_produtos", conexao )

In [23]:
print(df_resultado)

  id_produto                        nome_produto            categoria  \
0          1            Mapa de Uso do Solo 2023  Mapeamento Temático   
1          2          Modelo Digital de Elevação                  MDE   
2          3         Ortoimagem Urbana São Paulo       Imagem Orbital   
3          4       Mapa de Drenagem Hidrográfica          Hidrografia   
4          5  Classificação de Vegetação Cerrado            Vegetação   
5          6  Limites Administrativos Municipais    Base Cartográfica   
6          7       Mapa de Risco de Deslizamento    Análise Ambiental   

  tipo_dado resolucao_espacial    sistema_coordenadas  area_cobertura_km2  \
0  Vetorial                10m  SIRGAS 2000 / UTM 23S              1500.0   
1    Raster                30m                  WGS84              5000.0   
2    Raster               0.5m            SIRGAS 2000               800.0   
3  Vetorial            1:25000            SIRGAS 2000              3200.0   
4    Raster                20m

In [ ]:
# Sem formatação em dicionário:
resultado = conexao.execute(
    "SELECT * FROM bronze_produtos"
).fetchall()

for row in resultado:
    print(row)


('1', 'Mapa de Uso do Solo 2023', 'Mapeamento Temático', 'Vetorial', '10m', 'SIRGAS 2000 / UTM 23S', 1500.0, '2023-06-15', 'Shapefile', 'GeoMapas Ltda', 3500.0, '2025-12-22 00:38:37.735803')
('2', 'Modelo Digital de Elevação', 'MDE', 'Raster', '30m', 'WGS84', 5000.0, '2022-11-20', 'GeoTIFF', 'INPE', 0.0, '2025-12-22 00:38:37.735803')
('3', 'Ortoimagem Urbana São Paulo', 'Imagem Orbital', 'Raster', '0.5m', 'SIRGAS 2000', 800.0, '2023-02-10', 'GeoTIFF', 'Maxar', 12000.0, '2025-12-22 00:38:37.735803')
('4', 'Mapa de Drenagem Hidrográfica', 'Hidrografia', 'Vetorial', '1:25000', 'SIRGAS 2000', 3200.0, '2021-08-05', 'GeoPackage', 'ANA', 0.0, '2025-12-22 00:38:37.735803')
('5', 'Classificação de Vegetação Cerrado', 'Vegetação', 'Raster', '20m', 'WGS84', 2100.0, '2022-09-30', 'GeoTIFF', 'IBGE', 2500.0, '2025-12-22 00:38:37.735803')
('6', 'Limites Administrativos Municipais', 'Base Cartográfica', 'Vetorial', '1:50000', 'SIRGAS 2000', 8500.0, '2023-01-01', 'Shapefile', 'IBGE', 0.0, '2025-12-22 0

In [26]:
df_resultado.head()

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10m,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22 00:38:37.735803
1,2,Modelo Digital de Elevação,MDE,Raster,30m,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22 00:38:37.735803
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5m,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22 00:38:37.735803
3,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,1:25000,SIRGAS 2000,3200.0,2021-08-05,GeoPackage,ANA,0.0,2025-12-22 00:38:37.735803
4,5,Classificação de Vegetação Cerrado,Vegetação,Raster,20m,WGS84,2100.0,2022-09-30,GeoTIFF,IBGE,2500.0,2025-12-22 00:38:37.735803


In [29]:
# Por segurança , teremos outra tabela da bronze, com a mesma estrutura e os mesmo dados com nome diferente:
# CREATE TABLE bronze_z0019_backupe AS SELECT * FROM bronze_produtos;
# No SQLite3:

conexao.execute( """
CREATE TABLE bronze_z0019_backup AS SELECT * FROM bronze_produtos
                """)



In [31]:
# Validação dos dados em tela:

pd.read_sql("SELECT * FROM bronze_z0019_backup", conexao)


,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10m,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22 00:38:37.735803
1,2,Modelo Digital de Elevação,MDE,Raster,30m,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22 00:38:37.735803
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5m,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22 00:38:37.735803
3,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,1:25000,SIRGAS 2000,3200.0,2021-08-05,GeoPackage,ANA,0.0,2025-12-22 00:38:37.735803
4,5,Classificação de Vegetação Cerrado,Vegetação,Raster,20m,WGS84,2100.0,2022-09-30,GeoTIFF,IBGE,2500.0,2025-12-22 00:38:37.735803
5,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,1:50000,SIRGAS 2000,8500.0,2023-01-01,Shapefile,IBGE,0.0,2025-12-22 00:38:37.735803
6,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,1:10000,SIRGAS 2000 / UTM 22S,600.0,2023-07-12,GeoPackage,Defesa Civil,4800.0,2025-12-22 00:38:37.735803


In [ ]:
cursor = conexao.execute("""
SELECT name FROM sqlite_master
WHERE type = 'table'
ORDER BY name
    """)

tabelas = cursor.fetchall()
tabelas

# Assim temos duas tabelas no banco de dados. A mesma estrutura e os mesmo dados. Com nomes diferentes.

conexao.close()

[('bronze_produtos',), ('bronze_z0019_backup',)]